In [11]:
import pyspark

In [12]:
from pyspark import SparkContext
from pyspark.sql import SparkSession
import zipfile
import io

# Create a SparkSession first (this will also create SparkContext)
spark = SparkSession.builder.appName("ZipFileReader").getOrCreate()

# Get the SparkContext from the SparkSession
sc = spark.sparkContext

# Read the zip file as binary from a path (e.g., local, DBFS, S3, HDFS)
zipped_files = sc.binaryFiles("data/Divvy_Trips_2019_Q4.zip")  # Returns RDD of (path, binary)

def extract_csv_from_zip(partition):
    for file_path, content in partition:
        # Load binary content into BytesIO
        with zipfile.ZipFile(io.BytesIO(content), 'r') as z_file:
            # Assume you know the CSV filename inside the zip, e.g., 'data.csv'
            csv_filename = "data.csv"
            with z_file.open(csv_filename) as csv_file:
                # Read and decode CSV content line by line
                lines = csv_file.read().decode('utf-8').splitlines()
                # Skip header if needed, or yield all
                for line in lines:
                    yield line

# Apply the extraction and create an RDD of CSV lines
csv_lines_rdd = zipped_files.mapPartitions(extract_csv_from_zip)

# Convert RDD to DataFrame manually (you may need to parse fields)
from pyspark.sql.types import StructType, StructField, StringType
schema = StructType([StructField("col1", StringType()), StructField("col2", StringType())])  # Adjust as needed

# Convert lines to rows
from pyspark.sql import Row
rows_rdd = csv_lines_rdd.map(lambda line: Row(*line.split(","))).filter(lambda row: row[0] != 'header_col1')  # Skip header
df = spark.createDataFrame(rows_rdd, schema)

df.show()

25/11/04 16:28:16 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/04 16:28:18 ERROR Executor: Exception in task 0.0 in stage 0.0 (TID 0)/ 1]
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 2044, in main
    process()
    ~~~~~~~^^
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 2036, in process
    serializer.dump_stream(out_iter, outfile)
    ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/serializers.py", line 273, in dump_stream
    vs = list(itertools.islice(iterator, batch))
  File "/tmp/ipykernel_40724/721895367.py", line 21, in extract_csv_from_zip
  File "/home/developer/anaconda3/lib/python3.13/zipfile/__init__.py", line 1639,

Py4JJavaError: An error occurred while calling o186.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (192.168.29.9 executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 2044, in main
    process()
    ~~~~~~~^^
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 2036, in process
    serializer.dump_stream(out_iter, outfile)
    ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/serializers.py", line 273, in dump_stream
    vs = list(itertools.islice(iterator, batch))
  File "/tmp/ipykernel_40724/721895367.py", line 21, in extract_csv_from_zip
  Fi

In [ ]:
from pyspark import SparkContext
from pyspark.sql import SparkSession
import zipfile
import io

# Create a SparkSession
spark = SparkSession.builder.appName("MultipleZipFileReader").getOrCreate()
sc = spark.sparkContext

# Method 1: Using a wildcard pattern to read multiple zip files
# This will read all zip files in the data directory
zipped_files = sc.binaryFiles("data/*.zip")  # Returns RDD of (path, binary)

# Method 2: Explicitly specifying multiple paths
# zipped_files = sc.binaryFiles(["data/file1.zip", "data/file2.zip", "data/file3.zip"])

def extract_csv_from_zip(partition):
    for file_path, content in partition:
        try:
            with zipfile.ZipFile(io.BytesIO(content), 'r') as z_file:
                # Get list of all files in the zip
                file_list = z_file.namelist()
                
                # Process each CSV file in the zip
                for filename in file_list:
                    if filename.endswith('.csv'):  # Process only CSV files
                        with z_file.open(filename) as csv_file:
                            # Read and decode CSV content line by line
                            lines = csv_file.read().decode('utf-8').splitlines()
                            # Add source file information if needed
                            for line in lines:
                                # You can yield (source_zip, filename, line) if you need to track the source
                                yield (file_path, filename, line)
        except Exception as e:
            # Handle errors (log them or yield error information)
            yield (file_path, "ERROR", str(e))

# Apply the extraction and create an RDD of CSV lines with source information
csv_data_rdd = zipped_files.mapPartitions(extract_csv_from_zip)

# If you want to process just the CSV lines without source information
csv_lines_rdd = csv_data_rdd.map(lambda x: x[2])

# Convert to DataFrame (example)
# First, parse the CSV lines into fields
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType

# Adjust schema based on your CSV structure
schema = StructType([
    StructField("col1", StringType()), 
    StructField("col2", StringType())
    # Add more fields as needed
])

# Skip header rows and convert to Row objects
header = "header_col1,header_col2"  # Adjust based on your actual header
rows_rdd = csv_lines_rdd.filter(lambda line: line != header) \
                       .map(lambda line: Row(*line.split(",")))

# Create DataFrame
df = spark.createDataFrame(rows_rdd, schema)

# Show results
df.show()

# To create separate DataFrames for each CSV file:
# Group by source file
grouped_by_source = csv_data_rdd.map(lambda x: ((x[0], x[1]), x[2])).groupByKey().mapValues(list)

# Process each source separately
for (zip_path, csv_name), lines in grouped_by_source.collect():
    print(f"Processing {csv_name} from {zip_path}")
    # Create DataFrame for this specific file
    # (similar to above, but with the specific lines)

In [9]:
# Assistant
from pyspark import SparkContext
from pyspark.sql import SparkSession
import zipfile
import io

# Create a SparkSession first (this will also create SparkContext)
spark = SparkSession.builder.appName("ZipFileReader").getOrCreate()

# Get the SparkContext from the SparkSession
sc = spark.sparkContext

# Read the zip file as binary from a path
zipped_files = sc.binaryFiles("data/Divvy_Trips_2019_Q4.zip")  # Returns RDD of (path, binary)

def extract_csv_from_zip(partition):
    for file_path, content in partition:
        try:
            # Load binary content into BytesIO
            with zipfile.ZipFile(io.BytesIO(content), 'r') as z_file:
                # Get the first CSV file in the zip instead of assuming name
                csv_files = [f for f in z_file.namelist() if f.endswith('.csv')]
                if not csv_files:
                    continue  # Skip if no CSV files
                
                csv_filename = csv_files[0]  # Use the first CSV file found
                with z_file.open(csv_filename) as csv_file:
                    # Read and decode CSV content line by line
                    lines = csv_file.read().decode('utf-8').splitlines()
                    # Yield all lines
                    for line in lines:
                        yield line
        except Exception as e:
            # Add error handling to help diagnose issues
            yield f"Error processing {file_path}: {str(e)}"

# Apply the extraction and create an RDD of CSV lines
csv_lines_rdd = zipped_files.mapPartitions(extract_csv_from_zip)

# First, collect a sample to determine the schema
sample_lines = csv_lines_rdd.take(5)  # Take a few lines to inspect

# Assuming the first line is the header
if sample_lines:
    header = sample_lines[0].split(",")
    num_columns = len(header)
    
    # Create schema dynamically based on header
    from pyspark.sql.types import StructType, StructField, StringType
    schema = StructType([StructField(col.strip(), StringType()) for col in header])
    
    # Convert lines to rows, skipping the header
    from pyspark.sql import Row
    def parse_line(line):
        # Handle potential CSV parsing issues (quotes, commas in fields, etc.)
        parts = line.split(",")
        # Ensure we have the right number of columns
        if len(parts) == num_columns:
            return Row(*parts)
        else:
            # Handle mismatched columns (pad or truncate as needed)
            return Row(*(parts + [''] * (num_columns - len(parts)) if len(parts) < num_columns else parts[:num_columns]))
    
    # Filter out the header and parse the remaining lines
    rows_rdd = csv_lines_rdd.filter(lambda line: not line.startswith(sample_lines[0])).map(parse_line)
    
    # Create DataFrame
    df = spark.createDataFrame(rows_rdd, schema)
    
    # Show the result
    df.show()
else:
    print("No data found in the zip file")

25/11/04 16:27:53 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
[Stage 13:>                                                         (0 + 1) / 1]

+--------+-------------------+-------------------+------+------------+---------------+--------------------+--------------------+--------------------+--------------------+----------+---------+
| trip_id|         start_time|           end_time|bikeid|tripduration|from_station_id|   from_station_name|       to_station_id|     to_station_name|            usertype|    gender|birthyear|
+--------+-------------------+-------------------+------+------------+---------------+--------------------+--------------------+--------------------+--------------------+----------+---------+
|25223640|2019-10-01 00:01:39|2019-10-01 00:17:20|  2215|       940.0|             20|Sheffield Ave & K...|                 309|Leavitt St & Armi...|          Subscriber|      Male|     1987|
|25223641|2019-10-01 00:02:16|2019-10-01 00:06:34|  6328|       258.0|             19|Throop (Loomis) S...|                 241| Morgan St & Polk St|          Subscriber|      Male|     1998|
|25223642|2019-10-01 00:04:32|2019-10-01

Exception ignored in: <_io.BufferedWriter name=5>                               
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [10]:
from pyspark.sql import SparkSession
import zipfile
import io

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Read CSV from Zip") \
    .getOrCreate()

# Path to your zip file
zip_file_path = "data/"

def read_csv_from_zip(spark, zip_path):

    # Read the zip file as a binary file
    binary_file_rdd = spark.sparkContext.binaryFiles(zip_path)
    
    # Process each file in the zip
    def process_zip_file(file_tuple):
        file_path, file_content = file_tuple
        
        # Create a file-like object from the binary content
        zip_content = io.BytesIO(file_content)
        
        # Open the zip file
        with zipfile.ZipFile(zip_content) as z:
            # Find CSV files in the zip
            csv_files = [f for f in z.namelist() if f.lower().endswith('.csv')]
            
            if not csv_files:
                return []
            
            # Read the first CSV file found
            csv_file = csv_files[0]
            with z.open(csv_file) as f:
                # Convert to string and split into lines
                content = f.read().decode('utf-8')
                return content.splitlines()
    
    # Extract CSV lines from the zip file
    csv_lines = binary_file_rdd.flatMap(process_zip_file)
    
    # If there are lines, process them
    if csv_lines.count() > 0:
        # Get the header
        header = csv_lines.first()
        
        # Create a DataFrame from the CSV data
        csv_data = csv_lines.filter(lambda line: line != header)
        
        # Convert RDD to DataFrame using the header
        df = spark.read.csv(
            csv_data, 
            header=False, 
            inferSchema=True
        )
        
        # Rename columns using the header
        header_cols = header.split(',')
        for i, col_name in enumerate(header_cols):
            df = df.withColumnRenamed(f"_c{i}", col_name.strip())
        
        return df
    else:
        # Return empty DataFrame if no data
        return spark.createDataFrame([], "")

# Read the CSV from the zip file
df = read_csv_from_zip(spark, zip_file_path)

# Show the data
df.show()

# Stop the Spark session when done
spark.stop()

25/11/04 16:28:01 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
Exception ignored in: <_io.BufferedWriter name=5>                               
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe
Exception ignored in: <_io.BufferedWriter name=5>                               
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
| trip_id|         start_time|           end_time|bikeid|tripduration|from_station_id|   from_station_name|to_station_id|     to_station_name|  usertype|gender|birthyear|
+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
|25223640|2019-10-01 00:01:39|2019-10-01 00:17:20|  2215|       940.0|             20|Sheffield Ave & K...|          309|Leavitt St & Armi...|Subscriber|  Male|     1987|
|25223641|2019-10-01 00:02:16|2019-10-01 00:06:34|  6328|       258.0|             19|Throop (Loomis) S...|          241| Morgan St & Polk St|Subscriber|  Male|     1998|
|25223642|2019-10-01 00:04:32|2019-10-01 00:18:43|  3003|       850.0|             84|Milwaukee Ave & G...|          199|Wabash Ave & Gran...|Sub

In [7]:
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Read CSV from Zip") \
    .getOrCreate()

# Path to your zip file
zip_file_path = "data/Divvy_Trips_2019_Q4.zip"

# Read the CSV directly from the zip file using Spark's built-in functionality
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"zip://{zip_file_path}")

# Show the data
df.show()

# Stop the Spark session when done
spark.stop()

25/11/03 10:48:16 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: zip://data/Divvy_Trips_2019_Q4.zip.
org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "zip"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3581)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3612)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:172)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3716)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3667)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:55)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark

Py4JJavaError: An error occurred while calling o211.load.
: org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "zip"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3581)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3612)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:172)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3716)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3667)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:557)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:366)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:777)
	at scala.collection.immutable.List.map(List.scala:247)
	at scala.collection.immutable.List.map(List.scala:79)
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(DataSource.scala:775)
	at org.apache.spark.sql.executi

In [12]:
from pyspark.sql import SparkSession
import zipfile
import os
import tempfile

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Read CSV from Zip") \
    .getOrCreate()

# Path to your zip file
zip_file_path = "data/Divvy_Trips_2019_Q4.zip"

# Create a temporary directory to extract the zip file
temp_dir = tempfile.mkdtemp()

try:
    # Extract the zip file
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(temp_dir)
    
    # Find CSV files in the extracted directory
    csv_files = [os.path.join(temp_dir, f) for f in os.listdir(temp_dir) 
                if f.endswith('.csv')]
    
    if csv_files:
        # Read the first CSV file found (modify if you need a specific file)
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(csv_files[0])
        
        # Show the data
        df.count()
        df.show()
    else:
        print("No CSV files found in the zip archive")
        
finally:
    # Clean up the temporary directory
    import shutil
    shutil.rmtree(temp_dir)
    
    # Stop the Spark session when done
    

+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
| trip_id|         start_time|           end_time|bikeid|tripduration|from_station_id|   from_station_name|to_station_id|     to_station_name|  usertype|gender|birthyear|
+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
|25223640|2019-10-01 00:01:39|2019-10-01 00:17:20|  2215|       940.0|             20|Sheffield Ave & K...|          309|Leavitt St & Armi...|Subscriber|  Male|     1987|
|25223641|2019-10-01 00:02:16|2019-10-01 00:06:34|  6328|       258.0|             19|Throop (Loomis) S...|          241| Morgan St & Polk St|Subscriber|  Male|     1998|
|25223642|2019-10-01 00:04:32|2019-10-01 00:18:43|  3003|       850.0|             84|Milwaukee Ave & G...|          199|Wabash Ave & Gran...|Sub

In [9]:
type(df)

pyspark.sql.classic.dataframe.DataFrame

df.count()

# =====================================

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ReadZipCSV").getOrCreate()

df = spark.read.csv("data/Divvy_Trips_2019_Q4.zip", header=True, inferSchema=True)

df.show()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/04 16:20:17 WARN Utils: Your hostname, developer, resolves to a loopback address: 127.0.1.1; using 192.168.29.9 instead (on interface enp2s0)
25/11/04 16:20:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/04 16:20:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/04 16:20:19 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

+----------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|PK \b \b  �5P              Divvy_Trips_2019_Q4.csvUX\f

In [3]:
df.count()

142273

In [4]:
df1 = spark.read.csv("data/Divvy_Trips_2019_Q4.zip", header=True, inferSchema=True)

df1.show()


+----------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|PK \b \b  �5P              Divvy_Trips_2019_Q4.csvUX\f

In [8]:
df1.printSchema

<bound method DataFrame.printSchema of DataFrame[PK    �5P              Divvy_Trips_2019_Q4.csvUX
 �f'^Pf'^� ��]�
E�-�~���kp: string, O[����@@3ݨ[�p��R�DJ�T��U�F���^��#3|{xD�;f�5�B���s�����}|��؟������;ܽI�����}�@���y�p<��~;���7�a�
��ɿ���z8]~�o���������a����p��8?��t؟��6�v;ݩ���w�Y�=��s�?SṦ?������
�������v<ܾy���g�ﳿ��޿~<z����t���������+}�����a���?������������r{ة>��
U.G?W���n��
w�EZ��w��;�N�}�����x�U��_��nOg: string, H�_��t~��K��:����˥�1/E�_k���g*>��~�����D�{y�����pv���
���ݯ����w⣫�|x?: string, E奘K������}�w��bg����������q���oo�N���㧝���o�u��}��}8?{Qݜ.��VVT�̸紉N�v���E�"������n�_������{��|q���z�8��O_��ޞ��i����*K��yuϝکث�	W��݋����t/~=���E���T�����7ǻ�9�
nߟ�
���ՈKk���ȴ~N�V�Wؠ.hlP�ц��yq�x~K'D}w���^���	�Ŕ?�=^�ҁO��a}��=֛�O�9�?~��y�ˋ)Ϥsl]�b��w�wƚ�s���S;�
�\�ۢ�iӾ���=�-C�9=�Y����G>-
��]^�����É��n7: string, �	[잛�g&<��W;���k���@�����4����7����\�w?�?޾��V�`�C~RNXcr�3S�sGv��jl��q���� �b��������&}�������y��b68��tdyŭ�ϻnҥ�L���~�5��X
��W�+ǻ��)�8�g䅌�x�N��Xy

In [14]:
spark.stop()

In [15]:
import zipfile
import io
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ReadMultipleZipCSVs").getOrCreate()

# List of zipped files (modify paths as needed)
zip_files = ["data/Divvy_Trips_2019_Q4.zip", "data/Divvy_Trips_2020_Q1.zip"]

dataframes = []

for zip_file in zip_files:
    with zipfile.ZipFile(zip_file, "r") as z:
        # Since each ZIP has exactly 1 CSV
        for file in z.namelist():
            if file.endswith(".csv"):
                print(f"Reading file: {file} from {zip_file}")
                
                # 1. Read CSV content as decoded UTF-8 text (fixes weird output)
                data = z.read(file).decode("utf-8")
                
                # 2. Convert to list of rows & remove empty lines
                rows = [row for row in data.split("\n") if row.strip() != ""]
                
                # 3. Convert to Spark RDD
                rdd = spark.sparkContext.parallelize(rows)
                
                # 4. Read RDD into DataFrame
                df = spark.read.csv(rdd, header=True, inferSchema=True)
                
                dataframes.append(df)

# Merge all DataFrames into a single one
if dataframes:
    final_df = dataframes[0]
    for d in dataframes[1:]:
        final_df = final_df.unionByName(d, allowMissingColumns=True)
    
    print("✅ Final Merged DataFrame:")
    final_df.show(truncate=False)
    final_df.printSchema()
else:
    print("⚠ No CSV files found in ZIP archives.")


25/11/04 16:30:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Reading file: Divvy_Trips_2019_Q4.csv from data/Divvy_Trips_2019_Q4.zip


25/11/04 16:30:41 WARN TaskSetManager: Stage 0 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>                   (0 + 1) / 1]
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe
25/11/04 16:30:42 WARN TaskSetManager: Stage 1 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 1:>                                                          (0 + 4) / 4]

Reading file: __MACOSX/._Divvy_Trips_2019_Q4.csv from data/Divvy_Trips_2019_Q4.zip


UnicodeDecodeError: 'utf-8' codec can't decode byte 0x9c in position 45: invalid start byte

In [18]:
SparkSession.getActiveSession()

In [17]:
spark.stop()

In [1]:
import zipfile
import io
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ReadZipCSVs").getOrCreate()

zip_files = ["data/Divvy_Trips_2019_Q4.zip", "data/Divvy_Trips_2020_Q1.zip"]   # ← Add both zip paths

dataframes = []

for zip_file in zip_files:
    with zipfile.ZipFile(zip_file, "r") as z:
        for file in z.namelist():
            
            if file.startswith("__MACOSX") or not file.endswith(".csv"):
                continue

            print(f"Reading file: {file} from {zip_file}")

            # Read safely: ignore non-UTF-8 bytes
            try:
                data = z.read(file).decode("utf-8", errors="ignore")
            except:
                print(f"⚠ Skipping file (cannot decode): {file}")
                continue

            # Convert to rows and remove empty lines
            rows = [row for row in data.split("\n") if row.strip() != ""]

            # Convert to RDD
            rdd = spark.sparkContext.parallelize(rows)

            # Read into DataFrame
            df = spark.read.csv(rdd, header=True, inferSchema=True)
            dataframes.append(df)

# Combine all DataFrames
if dataframes:
    final_df = dataframes[0]
    for temp_df in dataframes[1:]:
        final_df = final_df.unionByName(temp_df, allowMissingColumns=True)

    final_df.show(5, truncate=False)
    final_df.printSchema()
else:
    print("⚠ No valid CSV files found.")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/06 10:31:53 WARN Utils: Your hostname, developer, resolves to a loopback address: 127.0.1.1; using 192.168.29.9 instead (on interface enp2s0)
25/11/06 10:31:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/06 10:31:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Reading file: Divvy_Trips_2019_Q4.csv from data/Divvy_Trips_2019_Q4.zip


25/11/06 10:32:08 WARN TaskSetManager: Stage 0 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>                   (0 + 1) / 1]
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe
25/11/06 10:32:10 WARN TaskSetManager: Stage 1 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Reading file: Divvy_Trips_2020_Q1.csv from data/Divvy_Trips_2020_Q1.zip


25/11/06 10:32:13 WARN TaskSetManager: Stage 2 contains a task of very large size (17622 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe
25/11/06 10:32:13 WARN TaskSetManager: Stage 3 contains a task of very large size (17622 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

+--------+-------------------+-------------------+------+------------+---------------+------------------------------+-------------+---------------------------+----------+------+---------+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+---------------+
|trip_id |start_time         |end_time           |bikeid|tripduration|from_station_id|from_station_name             |to_station_id|to_station_name            |usertype  |gender|birthyear|ride_id|rideable_type|started_at|ended_at|start_station_name|start_station_id|end_station_name|end_station_id|start_lat|start_lng|end_lat|end_lng|member_casual\r|
+--------+-------------------+-------------------+------+------------+---------------+------------------------------+-------------+---------------------------+----------+------+---------+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---

25/11/06 10:32:15 WARN TaskSetManager: Stage 4 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [20]:
final_df.count()

25/11/04 16:36:58 WARN TaskSetManager: Stage 5 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

1130941

In [25]:
final_df.show()

25/11/04 16:40:06 WARN TaskSetManager: Stage 8 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.


+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+---------------+
| trip_id|         start_time|           end_time|bikeid|tripduration|from_station_id|   from_station_name|to_station_id|     to_station_name|  usertype|gender|birthyear|ride_id|rideable_type|started_at|ended_at|start_station_name|start_station_id|end_station_name|end_station_id|start_lat|start_lng|end_lat|end_lng|member_casual\r|
+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+-------+-------------+----------+--------+------------------+----------------+----------------+--------------+---------+---------+-------+-------+---------------+
|

Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [26]:
final_df.columns

['trip_id',
 'start_time',
 'end_time',
 'bikeid',
 'tripduration',
 'from_station_id',
 'from_station_name',
 'to_station_id',
 'to_station_name',
 'usertype',
 'gender',
 'birthyear',
 'ride_id',
 'rideable_type',
 'started_at',
 'ended_at',
 'start_station_name',
 'start_station_id',
 'end_station_name',
 'end_station_id',
 'start_lat',
 'start_lng',
 'end_lat',
 'end_lng',
 'member_casual\r']

In [27]:
import pandas as pd

In [38]:
df11 = pd.read_csv("data/Divvy_Trips_2019_Q4.csv")

In [39]:
df22 = pd.read_csv("data/Divvy_Trips_2020_Q1.csv")

In [41]:
df11.shape

(704054, 12)

In [42]:
df22.shape

(426887, 13)

In [45]:
print(df22.columns)
df22.head()

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='object')


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,EACB19130B0CDA4A,docked_bike,2020-01-21 20:06:59,2020-01-21 20:14:30,Western Ave & Leland Ave,239,Clark St & Leland Ave,326.0,41.9665,-87.6884,41.9671,-87.6674,member
1,8FED874C809DC021,docked_bike,2020-01-30 14:22:39,2020-01-30 14:26:22,Clark St & Montrose Ave,234,Southport Ave & Irving Park Rd,318.0,41.9616,-87.6660,41.9542,-87.6644,member
2,789F3C21E472CA96,docked_bike,2020-01-09 19:29:26,2020-01-09 19:32:17,Broadway & Belmont Ave,296,Wilton Ave & Belmont Ave,117.0,41.9401,-87.6455,41.9402,-87.6530,member
3,C9A388DAC6ABF313,docked_bike,2020-01-06 16:17:07,2020-01-06 16:25:56,Clark St & Randolph St,51,Fairbanks Ct & Grand Ave,24.0,41.8846,-87.6319,41.8918,-87.6206,member
4,943BC3CBECCFD662,docked_bike,2020-01-30 08:37:16,2020-01-30 08:42:48,Clinton St & Lake St,66,Wells St & Hubbard St,212.0,41.8856,-87.6418,41.8899,-87.6343,member


In [ ]:
print(df11.columns)
df11.head()

In [48]:
print(len(a))
print(len(b))

13
12


In [54]:
a = [1,2,3,4,5,6,7,8,9,10]
b = pd.Series(a)

In [56]:
b.mean()

np.float64(5.5)

In [57]:
type(b)

pandas.core.series.Series

In [ ]:
df11['tripduration'].mean()

In [69]:
df11['usertype'].value_counts()

usertype
Subscriber    597860
Customer      106194
Name: count, dtype: int64

In [72]:
df11['start_time'].value_counts()

start_time
2019-12-05 17:11:39    7
2019-10-01 17:18:33    7
2019-10-08 17:06:07    6
2019-10-19 14:10:43    6
2019-10-18 08:15:05    6
                      ..
2019-10-20 14:05:27    1
2019-10-20 14:05:34    1
2019-10-20 14:05:46    1
2019-10-20 14:05:47    1
2019-12-31 23:57:17    1
Name: count, Length: 633380, dtype: int64

In [80]:
data = df11["start_time"].str.split('-')[2]

In [84]:
data[1]

'10'

In [82]:
pd.Series(data = df11["start_time"].str.split('-')[2])

0           2019
1             10
2    01 00:04:32
dtype: object

In [1]:
spark.stop()

NameError: name 'spark' is not defined

In [85]:
# ===============

In [3]:
import zipfile
import io
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("TwoZipTwoDFs")\
        .config("spark.sql.shuffle.partitions", "4")\
        .config("spark.driver.memory","4g")\
        .config("spark.executor.memory","2g")\
        .getOrCreate()

zip_files = [
    "data/Divvy_Trips_2019_Q4.zip",
    "data/Divvy_Trips_2020_Q1.zip"
]

dfs = {}   # Dictionary to store each DataFrame separately

for zip_file in zip_files:
    with zipfile.ZipFile(zip_file, "r") as z:
        for file in z.namelist():
            # Skip macOS or non-CSV files
            if file.startswith("__MACOSX") or not file.endswith(".csv"):
                continue

            print(f"Reading file: {file} from {zip_file}")

            # Read file safely (handle encoding issues)
            data = z.read(file).decode("utf-8", errors="ignore")

            # Convert the text to Spark RDD
            rows = [row for row in data.split("\n") if row.strip() != ""]
            rdd = spark.sparkContext.parallelize(rows)

            # Load as DataFrame
            df = spark.read.csv(rdd, header=True, inferSchema=True)

            # Save this dataframe with a name
            dfs[zip_file] = df

# Access DataFrames separately:
df_2019_Q4 = dfs.get("data/Divvy_Trips_2019_Q4.zip")
df_2020_Q1 = dfs.get("data/Divvy_Trips_2020_Q1.zip")

print("First ZIP DataFrame:")
df_2019_Q4.show(5, truncate=False)

print("Second ZIP DataFrame:")
df_2020_Q1.show(5, truncate=False)

25/11/06 10:36:35 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Reading file: Divvy_Trips_2019_Q4.csv from data/Divvy_Trips_2019_Q4.zip


25/11/06 10:36:36 WARN TaskSetManager: Stage 5 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe
25/11/06 10:36:36 WARN TaskSetManager: Stage 6 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Reading file: Divvy_Trips_2020_Q1.csv from data/Divvy_Trips_2020_Q1.zip


25/11/06 10:36:39 WARN TaskSetManager: Stage 7 contains a task of very large size (17622 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe
25/11/06 10:36:39 WARN TaskSetManager: Stage 8 contains a task of very large size (17622 KiB). The maximum recommended task size is 1000 KiB.
25/11/06 10:36:40 WARN TaskSetManager: Stage 9 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.


First ZIP DataFrame:


Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe
25/11/06 10:36:40 WARN TaskSetManager: Stage 10 contains a task of very large size (17622 KiB). The maximum recommended task size is 1000 KiB.


+--------+-------------------+-------------------+------+------------+---------------+------------------------------+-------------+---------------------------+----------+------+---------+
|trip_id |start_time         |end_time           |bikeid|tripduration|from_station_id|from_station_name             |to_station_id|to_station_name            |usertype  |gender|birthyear|
+--------+-------------------+-------------------+------+------------+---------------+------------------------------+-------------+---------------------------+----------+------+---------+
|25223640|2019-10-01 00:01:39|2019-10-01 00:17:20|2215  |940.0       |20             |Sheffield Ave & Kingsbury St  |309          |Leavitt St & Armitage Ave  |Subscriber|Male  |1987     |
|25223641|2019-10-01 00:02:16|2019-10-01 00:06:34|6328  |258.0       |19             |Throop (Loomis) St & Taylor St|241          |Morgan St & Polk St        |Subscriber|Male  |1998     |
|25223642|2019-10-01 00:04:32|2019-10-01 00:18:43|3003  |850

Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [4]:
df_2019_Q4.show(1)

+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
| trip_id|         start_time|           end_time|bikeid|tripduration|from_station_id|   from_station_name|to_station_id|     to_station_name|  usertype|gender|birthyear|
+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
|25223640|2019-10-01 00:01:39|2019-10-01 00:17:20|  2215|       940.0|             20|Sheffield Ave & K...|          309|Leavitt St & Armi...|Subscriber|  Male|     1987|
+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
only showing top 1 row


25/11/06 10:36:53 WARN TaskSetManager: Stage 11 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [207]:
df_2020_Q1.count()

25/11/05 15:36:40 WARN TaskSetManager: Stage 277 contains a task of very large size (17622 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

426887

In [9]:
a = df_2020_Q1.columns
len(a)

13

In [15]:
from pyspark.sql.functions import col

In [ ]:
df_2019_Q4.select(col("")).show(5)

In [39]:
new_data = df_2019_Q4.select(df_2019_Q4['start_time'])
new_data

DataFrame[start_time: timestamp]

In [42]:
from pyspark.sql.functions import col, to_date

In [45]:
new_data = new_data.withColumn('date_only', to_date(col('start_time')))

In [72]:
g_new = new_data.groupBy("date_only").agg()

In [73]:
g_new.sh

DataFrame[date_only: date, count: bigint]

In [63]:
ab = new_data.select(new_data["date_only"])

In [64]:
bc = ab.distinct()

In [65]:
cd = bc.count()

25/11/04 18:45:07 WARN TaskSetManager: Stage 32 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

In [68]:
cd

92

In [27]:
df_2019_Q4.describe()

DataFrame[summary: string, trip_id: string, bikeid: string, tripduration: string, from_station_id: string, from_station_name: string, to_station_id: string, to_station_name: string, usertype: string, gender: string, birthyear: string]

In [31]:
df_2019_Q4.printSchema()

root
 |-- trip_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- tripduration: string (nullable = true)
 |-- from_station_id: integer (nullable = true)
 |-- from_station_name: string (nullable = true)
 |-- to_station_id: integer (nullable = true)
 |-- to_station_name: string (nullable = true)
 |-- usertype: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthyear: integer (nullable = true)



In [28]:
df_2019_Q4.columns

['trip_id',
 'start_time',
 'end_time',
 'bikeid',
 'tripduration',
 'from_station_id',
 'from_station_name',
 'to_station_id',
 'to_station_name',
 'usertype',
 'gender',
 'birthyear']

In [29]:
df_2020_Q1.describe()

DataFrame[summary: string, ride_id: string, rideable_type: string, start_station_name: string, start_station_id: string, end_station_name: string, end_station_id: string, start_lat: string, start_lng: string, end_lat: string, end_lng: string, member_casual
: string]

In [2]:
df_2020_Q1.columns

['ride_id',
 'rideable_type',
 'started_at',
 'ended_at',
 'start_station_name',
 'start_station_id',
 'end_station_name',
 'end_station_id',
 'start_lat',
 'start_lng',
 'end_lat',
 'end_lng',
 'member_casual\r']

In [3]:
from pyspark.sql.functions import col

# Standardize column names
df_2019 = df_2019_Q4.withColumnRenamed("start_time", "start_time") \
                  .withColumnRenamed("tripduration", "trip_duration")

df_2020 = df_2020_Q1.withColumnRenamed("started_at", "start_time") \
                  .withColumnRenamed("ride_id", "trip_id") \
                  .withColumnRenamed("ended_at", "end_time")

In [5]:
df_2019.columns

['trip_id',
 'start_time',
 'end_time',
 'bikeid',
 'trip_duration',
 'from_station_id',
 'from_station_name',
 'to_station_id',
 'to_station_name',
 'usertype',
 'gender',
 'birthyear']

In [6]:
df_2020.columns

['trip_id',
 'rideable_type',
 'start_time',
 'end_time',
 'start_station_name',
 'start_station_id',
 'end_station_name',
 'end_station_id',
 'start_lat',
 'start_lng',
 'end_lat',
 'end_lng',
 'member_casual\r']

In [10]:
from pyspark.sql.functions import *

In [ ]:
output_path = f"reports/{report_name}"

# Write into a single part file
result.coalesce(1).write.csv(output_path, header=True, mode="overwrite")

In [41]:
# Q1. What is the average trip duration per day?
from pyspark.sql.functions import to_date, avg

def avg_trip_duration_per_day(df, report_name):
    df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
    result = df.withColumn("date", to_date(col("start_time"))) \
               .groupBy("date") \
               .agg(round(avg("Trip_Duration"),1).alias("avg_trip_duration"))
    
    result.write.csv(f"reports/{report_name}", header=True, mode="overwrite")
    return result
    result.coalesce(1).write.csv(output_path, header=True, mode="overwrite")

avg_trip_duration_per_day(df_2019_Q4, "Average_trip_Durtion")

25/11/05 11:39:23 WARN TaskSetManager: Stage 61 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

DataFrame[date: date, avg_trip_duration: double]

In [24]:
# Q2. How many trips were taken each day?
df_2019_Q4.columns

['trip_id',
 'start_time',
 'end_time',
 'bikeid',
 'tripduration',
 'from_station_id',
 'from_station_name',
 'to_station_id',
 'to_station_name',
 'usertype',
 'gender',
 'birthyear']

In [25]:
df = df_2019_Q4.withColumn("date_only", to_date(col("start_time")))
df.columns

['trip_id',
 'start_time',
 'end_time',
 'bikeid',
 'tripduration',
 'from_station_id',
 'from_station_name',
 'to_station_id',
 'to_station_name',
 'usertype',
 'gender',
 'birthyear',
 'date_only']

In [68]:
# Q2. How many trips were taken each day?
def total_trips(df):
    df = df.withColumn("date_only", to_date(col("start_time")))
    result = df.groupBy("date_only").agg(count("trip_id").alias("total_trips"))

    result.write.csv(f"reports/TotalTrips_rips_PerDay", header=True, mode="overwrite")
    return result

total_trips(df_2019_Q4)

25/11/05 12:08:10 WARN TaskSetManager: Stage 104 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

DataFrame[date_only: date, total_trips: bigint]

In [ ]:
# Q3. What was the most popular starting trip station for each month?

In [43]:
df_2019_Q4.columns

['trip_id',
 'start_time',
 'end_time',
 'bikeid',
 'tripduration',
 'from_station_id',
 'from_station_name',
 'to_station_id',
 'to_station_name',
 'usertype',
 'gender',
 'birthyear']

In [55]:
df_2019.groupBy("from_station_name").agg(count("trip_id").alias("Popular_Station")).agg(max("Popular_Station").alias("MostPopular")).show()

25/11/05 11:58:22 WARN TaskSetManager: Stage 86 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 86:>                                                         (0 + 4) / 4]

+-----------+
|MostPopular|
+-----------+
|      12937|
+-----------+



In [9]:
from pyspark.sql.window import Window

In [76]:
df = df_2019_Q4.withColumn("Month", month(col("start_time"))).groupBy("Month", "from_station_name").count()\
    .withColumn("Rank", row_number().over(Window.partitionBy("Month").orderBy(col("count").desc())))\
    .filter(col("Rank")==1).select("Month", "from_station_name", "count")

df.show()

25/11/05 12:15:49 WARN TaskSetManager: Stage 122 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 122:==============>                                          (1 + 3) / 4]

+-----+-------------------+-----+
|Month|  from_station_name|count|
+-----+-------------------+-----+
|   10|Canal St & Adams St| 6564|
|   11|Canal St & Adams St| 3445|
|   12|Canal St & Adams St| 2928|
+-----+-------------------+-----+



In [78]:
df1 = df_2019_Q4.withColumn("Month", month(col("start_time")))
df2 = df1.groupBy("Month", "from_station_name").count()
df3 = df2.withColumn("Rank", row_number().over(Window.partitionBy("Month").orderBy(col("count").desc())))
result = df3.filter(col("Rank")==1).select("Month", "from_station_name", "count")
result.show()

25/11/05 12:20:35 WARN TaskSetManager: Stage 128 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 128:>                                                        (0 + 4) / 4]

+-----+-------------------+-----+
|Month|  from_station_name|count|
+-----+-------------------+-----+
|   10|Canal St & Adams St| 6564|
|   11|Canal St & Adams St| 3445|
|   12|Canal St & Adams St| 2928|
+-----+-------------------+-----+



12937

In [79]:
# Q3. What was the most popular starting trip station for each month?
def Most_Popular_starting_station_each_month(df):
    df1 = df.withColumn("Month", month(col("start_time")))
    df2 = df1.groupBy("Month", "from_station_name").count()
    df3 = df2.withColumn("Rank", row_number().over(Window.partitionBy("Month").orderBy(col("count").desc())))
    result = df3.filter(col("Rank")==1).select("Month", "from_station_name", "count")

    result.write.csv(f"reports/Popular_Station_PerMonth", header=True, mode="overwrite")
    return result
    
Most_Popular_starting_station_each_month(df_2019_Q4)

25/11/05 12:21:24 WARN TaskSetManager: Stage 134 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

DataFrame[Month: int, from_station_name: string, count: bigint]

In [88]:
# Q4. What were the top 3 trip stations each day for the last two weeks?
df_2019_Q4.select(weekofyear("start_time")).show()

+----------------------+
|weekofyear(start_time)|
+----------------------+
|                    40|
|                    40|
|                    40|
|                    40|
|                    40|
|                    40|
|                    40|
|                    40|
|                    40|
|                    40|
|                    40|
|                    40|
|                    40|
|                    40|
|                    40|
|                    40|
|                    40|
|                    40|
|                    40|
|                    40|
+----------------------+
only showing top 20 rows


25/11/05 12:35:18 WARN TaskSetManager: Stage 143 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [81]:
df_2020_Q1.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual\r: string (nullable = true)



In [89]:
1500/60

25.0

In [90]:
df_2019_Q4.printSchema()

root
 |-- trip_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- tripduration: string (nullable = true)
 |-- from_station_id: integer (nullable = true)
 |-- from_station_name: string (nullable = true)
 |-- to_station_id: integer (nullable = true)
 |-- to_station_name: string (nullable = true)
 |-- usertype: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthyear: integer (nullable = true)



In [102]:
df = df_2019_Q4.select(to_date("start_time").alias("To_date"))
df.select(month("To_date")).show()

SyntaxError: expression cannot contain assignment, perhaps you meant "=="? (262991428.py, line 2)

In [125]:
df = df_2019_Q4.filter(to_date(col("start_time")) >= date_sub(current_date(), 14)) 
df.show()

25/11/05 13:16:58 WARN TaskSetManager: Stage 164 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
25/11/05 13:16:59 WARN TaskSetManager: Stage 165 contains a task of very large size (24032 KiB). The maximum recommended task size is 1000 KiB.
[Stage 165:>                                                        (0 + 3) / 3]

+-------+----------+--------+------+------------+---------------+-----------------+-------------+---------------+--------+------+---------+
|trip_id|start_time|end_time|bikeid|tripduration|from_station_id|from_station_name|to_station_id|to_station_name|usertype|gender|birthyear|
+-------+----------+--------+------+------------+---------------+-----------------+-------------+---------------+--------+------+---------+
+-------+----------+--------+------+------------+---------------+-----------------+-------------+---------------+--------+------+---------+



In [ ]:
two_weeks_ago = date_sub(current_timestamp(), 14)

df_last_2_weeks = df_2019_Q4.filter(F.col("from_station") >= two_weeks_ago)
df_last_2_weeks.show()

In [108]:
df_2019_Q4.show(5)

+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
| trip_id|         start_time|           end_time|bikeid|tripduration|from_station_id|   from_station_name|to_station_id|     to_station_name|  usertype|gender|birthyear|
+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
|25223640|2019-10-01 00:01:39|2019-10-01 00:17:20|  2215|       940.0|             20|Sheffield Ave & K...|          309|Leavitt St & Armi...|Subscriber|  Male|     1987|
|25223641|2019-10-01 00:02:16|2019-10-01 00:06:34|  6328|       258.0|             19|Throop (Loomis) S...|          241| Morgan St & Polk St|Subscriber|  Male|     1998|
|25223642|2019-10-01 00:04:32|2019-10-01 00:18:43|  3003|       850.0|             84|Milwaukee Ave & G...|          199|Wabash Ave & Gran...|Sub

25/11/05 12:55:48 WARN TaskSetManager: Stage 158 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [115]:
df_2019_Q4.select("from_station_name").show()

+--------------------+
|   from_station_name|
+--------------------+
|Sheffield Ave & K...|
|Throop (Loomis) S...|
|Milwaukee Ave & G...|
|Lakeview Ave & Fu...|
|Ashland Ave & Div...|
|Clark St & Wellin...|
|Milwaukee Ave & G...|
|Clark St & Wellin...|
|Clark St & Wellin...|
|Cottage Grove Ave...|
|Clinton St & Madi...|
|Green St & Madiso...|
|Clinton St & Lake St|
|Sheridan Rd & Irv...|
|Clinton St & Lake St|
|Kedzie Ave & Chic...|
|Kingsbury St & Er...|
|Wells St & Concor...|
|State St & Pearso...|
|Kingsbury St & Ki...|
+--------------------+
only showing top 20 rows


25/11/05 13:02:24 WARN TaskSetManager: Stage 159 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [118]:
df_2019_Q4.select("start_time").show()

+-------------------+
|         start_time|
+-------------------+
|2019-10-01 00:01:39|
|2019-10-01 00:02:16|
|2019-10-01 00:04:32|
|2019-10-01 00:04:32|
|2019-10-01 00:04:34|
|2019-10-01 00:04:38|
|2019-10-01 00:04:52|
|2019-10-01 00:04:57|
|2019-10-01 00:05:20|
|2019-10-01 00:05:20|
|2019-10-01 00:05:30|
|2019-10-01 00:07:25|
|2019-10-01 00:08:40|
|2019-10-01 00:08:52|
|2019-10-01 00:10:03|
|2019-10-01 00:10:46|
|2019-10-01 00:12:04|
|2019-10-01 00:12:47|
|2019-10-01 00:13:21|
|2019-10-01 00:15:07|
+-------------------+
only showing top 20 rows


25/11/05 13:03:34 WARN TaskSetManager: Stage 160 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [122]:
df = df_2019_Q4.withColumn("Date_wise", to_date("start_time"))
df1 = df.select("Date_wise")

In [123]:
df2 = df1.withColumn("Date", month("Date_wise"))
df2

+----------+----+
| Date_wise|Date|
+----------+----+
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
|2019-10-01|  10|
+----------+----+
only showing top 20 rows


25/11/05 13:07:26 WARN TaskSetManager: Stage 163 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [130]:
df = df_2019_Q4.select("from_station_id").distinct()
df.count()

25/11/05 13:20:30 WARN TaskSetManager: Stage 176 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

610

In [133]:
df_2019_Q4.select(last_day("start_time")).distinct().count()

25/11/05 13:26:28 WARN TaskSetManager: Stage 186 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

3

In [134]:
def top_3_stations_last_2_weeks(df):
    recent = df.filter(to_date(col("start_time")) >= date_sub(current_date(), 14)) \
               .withColumn("date", to_date(col("start_time"))) \
               .groupBy("date", "from_station_name") \
               .count()

    window = Window.partitionBy("date").orderBy(col("count").desc())

    result = recent.withColumn("rank", row_number().over(window)).filter(col("rank") <= 3)

    # result.write.csv(f"reports/Top3_stations", header=True, mode="overwrite")
    return result.show()

top_3_stations_last_2_weeks(df_2019_Q4)

25/11/05 13:35:05 WARN TaskSetManager: Stage 192 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 192:>                                                        (0 + 4) / 4]

+----+-----------------+-----+----+
|date|from_station_name|count|rank|
+----+-----------------+-----+----+
+----+-----------------+-----+----+



In [161]:
max_date = df.select(F.max('date').alias('max_date')).first()[0]
max_date


25/11/05 14:56:12 WARN TaskSetManager: Stage 216 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

datetime.date(2019, 12, 31)

In [162]:
two_weeks_before = max_date-timedelta(weeks=2)

In [163]:
two_weeks_before

datetime.date(2019, 12, 17)

In [173]:
# Filter rows
filtered_df = df_2019_Q4.filter(df_2019_Q4.start_time.between(two_weeks_before, max_date))
filtered_df.show()

25/11/05 15:13:34 WARN TaskSetManager: Stage 234 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
25/11/05 15:13:35 WARN TaskSetManager: Stage 235 contains a task of very large size (24032 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>                   (0 + 3) / 3]
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe
[Stage 235:===================>                                     (1 + 2) / 3]

+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
| trip_id|         start_time|           end_time|bikeid|tripduration|from_station_id|   from_station_name|to_station_id|     to_station_name|  usertype|gender|birthyear|
+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
|25896951|2019-12-17 00:02:45|2019-12-17 00:10:40|  3958|       474.0|             48|Larrabee St & Kin...|           91|Clinton St & Wash...|Subscriber|  Male|     1973|
|25896952|2019-12-17 00:07:55|2019-12-17 00:13:42|  2155|       346.0|            338|Calumet Ave & 18t...|          255|Indiana Ave & Roo...|Subscriber|  Male|     1990|
|25896953|2019-12-17 00:08:06|2019-12-17 00:21:58|  2370|       832.0|            106|State St & Pearso...|          289|Wells St & Concor...|Sub

In [182]:
filtered1 = filtered_df.withColumn("Date", to_date(col("start_time")))
filtered2 = filtered1.groupBy("Date", "from_station_name").count()

In [184]:
filtered2.show()

25/11/05 15:18:07 WARN TaskSetManager: Stage 236 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 236:>                                                        (0 + 4) / 4]

+----------+--------------------+-----+
|      Date|   from_station_name|count|
+----------+--------------------+-----+
|2019-12-17|Calumet Ave & 18t...|   13|
|2019-12-17|Dearborn St & Eri...|   45|
|2019-12-17|Wells St & Concor...|   32|
|2019-12-17|Ashland Ave & Chi...|   10|
|2019-12-17|Clark St & Winnem...|   15|
|2019-12-17|Michigan Ave & Pe...|   17|
|2019-12-17|Franklin St & Chi...|   37|
|2019-12-17|Campbell Ave & Mo...|    4|
|2019-12-17|Western Ave & Fil...|    2|
|2019-12-17|State St & Randol...|   40|
|2019-12-17|Clybourn Ave & Di...|   12|
|2019-12-17|Southport Ave & I...|   12|
|2019-12-17| Wabash Ave & 9th St|   20|
|2019-12-17|Canal St & Madiso...|   69|
|2019-12-17|Lincoln Ave & Div...|   10|
|2019-12-17|Leavitt St & Chic...|    7|
|2019-12-17|Clark St & Wright...|   17|
|2019-12-17|Winthrop Ave & La...|    2|
|2019-12-17|Clark St & Wellin...|   18|
|2019-12-17| MLK Jr Dr & 29th St|    5|
+----------+--------------------+-----+
only showing top 20 rows


In [187]:
window = Window.partitionBy("Date").orderBy(col("count").desc())

result = filtered2.withColumn("rank", row_number().over(window)).filter(col("rank") <= 3)
result.orderBy("rank").count()

25/11/05 15:20:08 WARN TaskSetManager: Stage 251 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

42

In [ ]:
filter_data = df.filter((col("date") > two_weeks_before) & (col("date") <= max_date))   

In [221]:
# Q4. What were the top 3 trip stations each day for the last two weeks?
def top3_trip_stations_each_day_for_last_2_weeks(df):
    df = df.withColumn("date", to_date("start_time"))
    
    max_date = df.select(F.max('date').alias('max_date')).first()[0]
    two_weeks_before = max_date-timedelta(weeks=2)

    filter_data = df.filter((col("date") > two_weeks_before) & (col("date") <= max_date)) \
                .groupBy("date", "from_station_name") \
                .count()

    window = Window.partitionBy("Date").orderBy(col("count").desc())


    result = filter_data.withColumn("rank", row_number().over(window)) \
                    .filter(col("rank") <= 3) \
                    .orderBy("date", "rank")

    result.write.csv(f"reports/top3_trip_stations_last_2_week", header=True, mode="overwrite")
    return result

top3_trip_stations_each_day_for_last_2_weeks(df_2019_Q4)

25/11/05 16:14:37 WARN TaskSetManager: Stage 366 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
25/11/05 16:14:39 WARN TaskSetManager: Stage 369 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

DataFrame[date: date, from_station_name: string, count: bigint, rank: int]

In [8]:
from pyspark.sql import functions as F

In [244]:
# Q5. Do Males or Females take longer trips on average?

def trips_avg_of_male_female(df):
    updated_df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
    result = updated_df.groupBy("gender").agg(round(avg("Trip_Duration"),1).alias("Average"))

    result.write.csv(f"reports/trips_avg_of_male_female", header=True, mode="overwrite")
    return result

trips_avg_of_male_female(df_2019_Q4)

25/11/05 16:35:43 WARN TaskSetManager: Stage 399 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

DataFrame[gender: string, Average: double]

In [289]:
df_2019_Q4.show()

+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
| trip_id|         start_time|           end_time|bikeid|tripduration|from_station_id|   from_station_name|to_station_id|     to_station_name|  usertype|gender|birthyear|
+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
|25223640|2019-10-01 00:01:39|2019-10-01 00:17:20|  2215|       940.0|             20|Sheffield Ave & K...|          309|Leavitt St & Armi...|Subscriber|  Male|     1987|
|25223641|2019-10-01 00:02:16|2019-10-01 00:06:34|  6328|       258.0|             19|Throop (Loomis) S...|          241| Morgan St & Polk St|Subscriber|  Male|     1998|
|25223642|2019-10-01 00:04:32|2019-10-01 00:18:43|  3003|       850.0|             84|Milwaukee Ave & G...|          199|Wabash Ave & Gran...|Sub

25/11/05 18:45:07 WARN TaskSetManager: Stage 416 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [290]:
df1 = df_2019_Q4

In [291]:
df1.printSchema()

root
 |-- trip_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- tripduration: string (nullable = true)
 |-- from_station_id: integer (nullable = true)
 |-- from_station_name: string (nullable = true)
 |-- to_station_id: integer (nullable = true)
 |-- to_station_name: string (nullable = true)
 |-- usertype: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthyear: integer (nullable = true)



In [237]:
df1 = df1.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))

In [242]:
df1.groupBy("gender").agg(round(avg("Trip_Duration"),1).alias("Average")).show()

25/11/05 16:33:29 WARN TaskSetManager: Stage 396 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 396:>                                                        (0 + 4) / 4]

+------+-------+
|gender|Average|
+------+-------+
|Female| 1102.0|
|  NULL| 3826.6|
|  Male|  855.3|
+------+-------+



In [240]:
df1.filter(col("gender").isNotNull()).groupBy("gender").agg(avg("Trip_Duration")).show()

25/11/05 16:30:51 WARN TaskSetManager: Stage 390 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 390:>                                                        (0 + 4) / 4]

+------+------------------+
|gender|avg(Trip_Duration)|
+------+------------------+
|Female|1101.9518158459377|
|  Male| 855.3141806400951|
+------+------------------+



In [233]:
df1.gender.isNull

Column<'isNull(gender)'>

In [ ]:
# Q5. Do Males or Females take longer trips on average?

def trips_avg_of_male_female(df):
    updated_df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
    result = updated_df.groupBy("gender").agg(round(avg("Trip_Duration"),1).alias("Average"))

    result.write.csv(f"reports/trips_avg_of_male_female", header=True, mode="overwrite")
    return result

trips_avg_of_male_female(df_2019_Q4)

In [245]:
# Q6. What is the top 10 ages of those that take the longest trips, and shortest?

In [20]:
df1 = df_2019_Q4

In [21]:
df2 = df1
# df2.select("birthyear").show()
df2.show()

+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
| trip_id|         start_time|           end_time|bikeid|tripduration|from_station_id|   from_station_name|to_station_id|     to_station_name|  usertype|gender|birthyear|
+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
|25223640|2019-10-01 00:01:39|2019-10-01 00:17:20|  2215|       940.0|             20|Sheffield Ave & K...|          309|Leavitt St & Armi...|Subscriber|  Male|     1987|
|25223641|2019-10-01 00:02:16|2019-10-01 00:06:34|  6328|       258.0|             19|Throop (Loomis) S...|          241| Morgan St & Polk St|Subscriber|  Male|     1998|
|25223642|2019-10-01 00:04:32|2019-10-01 00:18:43|  3003|       850.0|             84|Milwaukee Ave & G...|          199|Wabash Ave & Gran...|Sub

25/11/06 10:44:13 WARN TaskSetManager: Stage 20 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [22]:
df2 = df2.withColumn("Year_of_Birth", to_date("birthyear"))
df2.show()

+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+-------------+
| trip_id|         start_time|           end_time|bikeid|tripduration|from_station_id|   from_station_name|to_station_id|     to_station_name|  usertype|gender|birthyear|Year_of_Birth|
+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+-------------+
|25223640|2019-10-01 00:01:39|2019-10-01 00:17:20|  2215|       940.0|             20|Sheffield Ave & K...|          309|Leavitt St & Armi...|Subscriber|  Male|     1987|   1987-01-01|
|25223641|2019-10-01 00:02:16|2019-10-01 00:06:34|  6328|       258.0|             19|Throop (Loomis) S...|          241| Morgan St & Polk St|Subscriber|  Male|     1998|   1998-01-01|
|25223642|2019-10-01 00:04:32|2019-10-01 00:18:43|  3003|       850.0|     

25/11/06 10:44:16 WARN TaskSetManager: Stage 21 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [23]:
df2.select("Year_of_Birth").show()

+-------------+
|Year_of_Birth|
+-------------+
|   1987-01-01|
|   1998-01-01|
|   1991-01-01|
|   1990-01-01|
|   1987-01-01|
|   1994-01-01|
|   1991-01-01|
|   1995-01-01|
|   1993-01-01|
|         NULL|
|   1977-01-01|
|   1980-01-01|
|   1994-01-01|
|         NULL|
|   1992-01-01|
|         NULL|
|   1962-01-01|
|   1992-01-01|
|         NULL|
|   1995-01-01|
+-------------+
only showing top 20 rows


25/11/06 10:44:27 WARN TaskSetManager: Stage 22 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [24]:
df2 = df2.withColumn("Age", floor(date_diff(F.current_date(), F.col("Year_of_Birth"))/365.25))
df2.select("Age").show()

+----+
| Age|
+----+
|  38|
|  27|
|  34|
|  35|
|  38|
|  31|
|  34|
|  30|
|  32|
|NULL|
|  48|
|  45|
|  31|
|NULL|
|  33|
|NULL|
|  63|
|  33|
|NULL|
|  30|
+----+
only showing top 20 rows


25/11/06 10:44:31 WARN TaskSetManager: Stage 23 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [25]:
df2.printSchema()

root
 |-- trip_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- tripduration: string (nullable = true)
 |-- from_station_id: integer (nullable = true)
 |-- from_station_name: string (nullable = true)
 |-- to_station_id: integer (nullable = true)
 |-- to_station_name: string (nullable = true)
 |-- usertype: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthyear: integer (nullable = true)
 |-- Year_of_Birth: date (nullable = true)
 |-- Age: long (nullable = true)



In [293]:
df2.show()

+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
| trip_id|         start_time|           end_time|bikeid|tripduration|from_station_id|   from_station_name|to_station_id|     to_station_name|  usertype|gender|birthyear|
+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+
|25223640|2019-10-01 00:01:39|2019-10-01 00:17:20|  2215|       940.0|             20|Sheffield Ave & K...|          309|Leavitt St & Armi...|Subscriber|  Male|     1987|
|25223641|2019-10-01 00:02:16|2019-10-01 00:06:34|  6328|       258.0|             19|Throop (Loomis) S...|          241| Morgan St & Polk St|Subscriber|  Male|     1998|
|25223642|2019-10-01 00:04:32|2019-10-01 00:18:43|  3003|       850.0|             84|Milwaukee Ave & G...|          199|Wabash Ave & Gran...|Sub

25/11/05 18:45:49 WARN TaskSetManager: Stage 418 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [295]:
df2.printSchema()

root
 |-- trip_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- tripduration: string (nullable = true)
 |-- from_station_id: integer (nullable = true)
 |-- from_station_name: string (nullable = true)
 |-- to_station_id: integer (nullable = true)
 |-- to_station_name: string (nullable = true)
 |-- usertype: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthyear: integer (nullable = true)



In [303]:
df2 = df2.withColumn("Year_birth", to_date("birthyear"))

In [305]:
df2.select("Year_birth").show()

+----------+
|Year_birth|
+----------+
|1987-01-01|
|1998-01-01|
|1991-01-01|
|1990-01-01|
|1987-01-01|
|1994-01-01|
|1991-01-01|
|1995-01-01|
|1993-01-01|
|      NULL|
|1977-01-01|
|1980-01-01|
|1994-01-01|
|      NULL|
|1992-01-01|
|      NULL|
|1962-01-01|
|1992-01-01|
|      NULL|
|1995-01-01|
+----------+
only showing top 20 rows


25/11/05 18:50:33 WARN TaskSetManager: Stage 421 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [29]:
updated_df = df2.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
updated_df.show()

+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+-------------+----+-------------+
| trip_id|         start_time|           end_time|bikeid|tripduration|from_station_id|   from_station_name|to_station_id|     to_station_name|  usertype|gender|birthyear|Year_of_Birth| Age|Trip_Duration|
+--------+-------------------+-------------------+------+------------+---------------+--------------------+-------------+--------------------+----------+------+---------+-------------+----+-------------+
|25223640|2019-10-01 00:01:39|2019-10-01 00:17:20|  2215|       940.0|             20|Sheffield Ave & K...|          309|Leavitt St & Armi...|Subscriber|  Male|     1987|   1987-01-01|  38|        940.0|
|25223641|2019-10-01 00:02:16|2019-10-01 00:06:34|  6328|       258.0|             19|Throop (Loomis) S...|          241| Morgan St & Polk St|Subscriber|  Male|     1998|   1998-01-01|

25/11/06 10:49:47 WARN TaskSetManager: Stage 30 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [48]:
updated_df.filter(F.col("Age").isNotNull()).groupBy("Age", "Trip_Duration").count().orderBy(F.desc("Trip_Duration")).show(10)
# updated_df.filter(F.col("Age").isNotNull()).groupBy("Age", "Trip_Duration").count().orderBy(F.desc("Trip_Duration")).show(10)


25/11/06 10:59:37 WARN TaskSetManager: Stage 58 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 58:>                                                         (0 + 4) / 4]

+---+-------------+-----+
|Age|Trip_Duration|count|
+---+-------------+-----+
| 29|       9994.0|    1|
| 74|        999.0|    1|
| 36|        999.0|   10|
| 60|        999.0|    4|
| 50|        999.0|    1|
| 40|        999.0|    7|
| 38|        999.0|   12|
| 49|        999.0|    6|
| 30|        999.0|    6|
| 63|        999.0|    2|
+---+-------------+-----+
only showing top 10 rows


In [49]:
# Q6. What is the top 10 ages of those that take the longest trips, and shortest?
updated_df.filter(F.col("Age").isNotNull()).groupBy("Age", "Trip_Duration").count().orderBy(F.desc("Trip_Duration")).show(10)

25/11/06 11:00:33 WARN TaskSetManager: Stage 61 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 61:==============>                                           (1 + 3) / 4]

+---+-------------+-----+
|Age|Trip_Duration|count|
+---+-------------+-----+
| 29|       9994.0|    1|
| 74|        999.0|    1|
| 36|        999.0|   10|
| 60|        999.0|    4|
| 50|        999.0|    1|
| 40|        999.0|    7|
| 38|        999.0|   12|
| 49|        999.0|    6|
| 30|        999.0|    6|
| 63|        999.0|    2|
+---+-------------+-----+
only showing top 10 rows


In [50]:
updated_df.filter(F.col("Age").isNotNull()).groupBy("Age", "Trip_Duration").count().orderBy("Trip_Duration").show(10)

25/11/06 11:01:18 WARN TaskSetManager: Stage 64 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 64:=============================>                            (2 + 2) / 4]

+---+-------------+-----+
|Age|Trip_Duration|count|
+---+-------------+-----+
| 36|        100.0|   11|
| 63|        100.0|    3|
| 38|        100.0|   14|
| 28|        100.0|    2|
| 35|        100.0|    9|
| 39|        100.0|    5|
| 58|        100.0|    2|
| 27|        100.0|    5|
| 37|        100.0|    6|
| 29|        100.0|    6|
+---+-------------+-----+
only showing top 10 rows


In [53]:
# Q6. What is the top 10 ages of those that take the longest trips, and shortest?

def top_ten_ages_with_shortest_and_longest_trips(df):
    df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
    df = df.withColumn("Year_of_Birth", to_date("birthyear"))
    df = df.withColumn("Age", floor(date_diff(F.current_date(), F.col("Year_of_Birth"))/365.25))

    result1 = df.filter(F.col("Age").isNotNull()).groupBy("Age", "Trip_Duration").count().orderBy(F.desc("Trip_Duration")).show(10)
    result2 = df.filter(F.col("Age").isNotNull()).groupBy("Age", "Trip_Duration").count().orderBy("Trip_Duration").show(10)

    result1.show()
    result2.show()
    return result1, result2


top_ten_ages_with_shortest_and_longest_trips(df_2019_Q4)

25/11/06 11:11:14 WARN TaskSetManager: Stage 79 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

+---+-------------+-----+
|Age|Trip_Duration|count|
+---+-------------+-----+
| 29|       9994.0|    1|
| 74|        999.0|    1|
| 36|        999.0|   10|
| 60|        999.0|    4|
| 50|        999.0|    1|
| 40|        999.0|    7|
| 38|        999.0|   12|
| 49|        999.0|    6|
| 30|        999.0|    6|
| 63|        999.0|    2|
+---+-------------+-----+
only showing top 10 rows


25/11/06 11:11:16 WARN TaskSetManager: Stage 82 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 82:==============>                                           (1 + 3) / 4]

+---+-------------+-----+
|Age|Trip_Duration|count|
+---+-------------+-----+
| 36|        100.0|   11|
| 63|        100.0|    3|
| 38|        100.0|   14|
| 28|        100.0|    2|
| 35|        100.0|    9|
| 39|        100.0|    5|
| 58|        100.0|    2|
| 27|        100.0|    5|
| 37|        100.0|    6|
| 29|        100.0|    6|
+---+-------------+-----+
only showing top 10 rows


AttributeError: 'NoneType' object has no attribute 'show'

In [134]:
# Q6. What is the top 10 ages of those that take the longest trips, and shortest?

def top_ten_ages_with_shortest_and_longest_trips(df):
    df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
    df = df.withColumn("Year_of_Birth", to_date("birthyear"))
    df = df.withColumn("Age", floor(date_diff(F.current_date(), F.col("Year_of_Birth"))/365.25))

    # Store the results in variables without showing them yet
    longest_trips  = df.filter(F.col("Age").isNotNull()).orderBy(col("Trip_Duration").desc()).select("Age", "Trip_Duration").limit(10)
    shortest_trips = df.filter(F.col("Age").isNotNull()).orderBy(col("Trip_Duration").asc()).select("Age", "Trip_Duration").limit(10)

    longest_trips.write.csv(f"reports/Top_Ten_Ages/Longest_trip", header=True, mode="overwrite")

    shortest_trips.write.csv(f"reports/Top_Ten_Ages/Shortest_trip", header=True, mode="overwrite")
    
    return "Result File for Question 6 Downloaded Successfully!!"

top_ten_ages_with_shortest_and_longest_trips(df_2019_Q4)

25/11/06 12:25:53 WARN TaskSetManager: Stage 186 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
25/11/06 12:25:55 WARN TaskSetManager: Stage 188 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

'Result File for Question 6 Downloaded Successfully!!'

In [ ]:
updated_df.orderBy("Trip_Duration").show()

In [69]:
updated_df.filter(F.col("Age").isNotNull()).groupBy("Age", "Trip_Duration").count().orderBy("Age").limit(10).show()


25/11/06 11:33:58 WARN TaskSetManager: Stage 104 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 104:>                                                        (0 + 4) / 4]

+---+-------------+-----+
|Age|Trip_Duration|count|
+---+-------------+-----+
| 22|        543.0|    1|
| 22|       1070.0|    1|
| 22|       1496.0|    1|
| 22|       1061.0|    1|
| 22|       7235.0|    1|
| 22|        740.0|    2|
| 22|        836.0|    1|
| 22|       1170.0|    1|
| 22|        848.0|    1|
| 22|        856.0|    1|
+---+-------------+-----+



In [70]:
updated_df.filter(F.col("Age").isNotNull()).groupBy("Age", "Trip_Duration").agg(max("Trip_Duration")).orderBy("Age").limit(10).show()

25/11/06 11:34:32 WARN TaskSetManager: Stage 107 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

+---+-------------+------------------+
|Age|Trip_Duration|max(Trip_Duration)|
+---+-------------+------------------+
| 22|       1044.0|            1044.0|
| 22|       1027.0|            1027.0|
| 22|       1098.0|            1098.0|
| 22|       1058.0|            1058.0|
| 22|       1283.0|            1283.0|
| 22|       1061.0|            1061.0|
| 22|       1284.0|            1284.0|
| 22|       1070.0|            1070.0|
| 22|       1427.0|            1427.0|
| 22|       1102.0|            1102.0|
+---+-------------+------------------+



In [77]:
updated_df = updated_df.filter(F.col("Trip_Duration").isNotNull())

In [86]:
updated_df.select("Trip_Duration")

False

In [ ]:
updated_df.filter(F.col("Age").isNotNull()).groupBy("Age").agg(max("Trip_Duration")).orderBy("Age", "Trip_Duration").limit(10).show()

In [72]:
updated_df.select("Trip_Duration").agg(max("Trip_Duration")).show()

25/11/06 11:37:00 WARN TaskSetManager: Stage 113 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 113:>                                                        (0 + 4) / 4]

+------------------+
|max(Trip_Duration)|
+------------------+
|            9998.0|
+------------------+



In [73]:
updated_df.select("tripduration").agg(max("tripduration")).show()

25/11/06 11:37:21 WARN TaskSetManager: Stage 116 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 116:>                                                        (0 + 4) / 4]

+-----------------+
|max(tripduration)|
+-----------------+
|            999.0|
+-----------------+



In [95]:
a = "12.4"
b = float(a)+3
b

15.4

In [104]:
res = updated_df.filter(F.col("Age").isNotNull()).groupBy("Age").agg(max("Trip_Duration").alias("Longest")).orderBy("Age").limit(10)
res.show()

25/11/06 11:57:17 WARN TaskSetManager: Stage 143 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 143:>                                                        (0 + 4) / 4]

+---+-------+
|Age|Longest|
+---+-------+
| 22|  983.0|
| 23|  992.0|
| 24|  998.0|
| 25|  998.0|
| 26|  999.0|
| 27|  999.0|
| 28|  999.0|
| 29| 9994.0|
| 30|  999.0|
| 31|  999.0|
+---+-------+



In [107]:
res2 = updated_df.filter(F.col("Age").isNotNull()).groupBy("Age").agg(max("Trip_Duration").alias("Longest")).(orderBy(desc("Age")).limit(10)
res2.show()

SyntaxError: invalid syntax (36546764.py, line 1)

In [112]:
# Corrected code
res2 = updated_df.filter(F.col("Age").isNotNull())\
        .groupBy("Age")\
        .agg(max("Trip_Duration").alias("Shortest"))\
        .orderBy(desc("Age"))\
        .limit(10)
res2.show()

25/11/06 12:04:17 WARN TaskSetManager: Stage 152 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 152:>                                                        (0 + 4) / 4]

+---+--------+
|Age|Shortest|
+---+--------+
| 22|   983.0|
| 23|   992.0|
| 24|   998.0|
| 25|   998.0|
| 26|   999.0|
| 27|   999.0|
| 28|   999.0|
| 29|  9994.0|
| 30|   999.0|
| 31|   999.0|
+---+--------+



In [117]:
updated_df.select("Trip_Duration").orderBy("Trip_Duration").distinct().limit(10).show()

25/11/06 12:06:56 WARN TaskSetManager: Stage 162 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 162:==============>                                          (1 + 3) / 4]

+-------------+
|Trip_Duration|
+-------------+
|       1867.0|
|        373.0|
|       1072.0|
|       1458.0|
|       1437.0|
|       8306.0|
|       1925.0|
|       1224.0|
|       1496.0|
|        761.0|
+-------------+



In [126]:
longest = updated_df.filter(F.col("Age").isNotNull()).orderBy(col("Trip_Duration").desc()).select("Age", "Trip_Duration").limit(10)
longest.show()

25/11/06 12:14:22 WARN TaskSetManager: Stage 170 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 170:>                                                        (0 + 4) / 4]

+---+-------------+
|Age|Trip_Duration|
+---+-------------+
| 29|       9994.0|
| 40|        999.0|
| 60|        999.0|
| 40|        999.0|
| 32|        999.0|
| 33|        999.0|
| 34|        999.0|
| 36|        999.0|
| 34|        999.0|
| 38|        999.0|
+---+-------------+



In [128]:
shortest = updated_df.filter(F.col("Age").isNotNull()).orderBy(col("Trip_Duration").asc()).select("Age", "Trip_Duration").limit(10)
shortest.show()

25/11/06 12:14:48 WARN TaskSetManager: Stage 171 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
[Stage 171:>                                                        (0 + 4) / 4]

+---+-------------+
|Age|Trip_Duration|
+---+-------------+
| 28|        100.0|
| 63|        100.0|
| 38|        100.0|
| 35|        100.0|
| 47|        100.0|
| 29|        100.0|
| 35|        100.0|
| 30|        100.0|
| 30|        100.0|
| 24|        100.0|
+---+-------------+



In [145]:
# ====================================================
import zipfile
import io
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import timedelta

def main():

    spark = SparkSession.builder.appName("Exercise6").enableHiveSupport().getOrCreate()
    
    zip_files = [
        "data/Divvy_Trips_2019_Q4.zip",
        "data/Divvy_Trips_2020_Q1.zip"
    ]
    
    dfs = {}   
    
    for zip_file in zip_files:
        with zipfile.ZipFile(zip_file, "r") as z:
            for file in z.namelist():
                # Skip macOS or non-CSV files
                if file.startswith("__MACOSX") or not file.endswith(".csv"):
                    continue
    
                print(f"Reading file: {file} from {zip_file}")
    
                # Read file safely (handle encoding issues)
                data = z.read(file).decode("utf-8", errors="ignore")
    
                # Convert the text to Spark RDD
                rows = [row for row in data.split("\n") if row.strip() != ""]
                rdd = spark.sparkContext.parallelize(rows)
    
                # Load as DataFrame
                df = spark.read.csv(rdd, header=True, inferSchema=True)
    
                # Save this dataframe with a name
                dfs[zip_file] = df
    
    # Access DataFrames separately:
    df1 = dfs.get("data/Divvy_Trips_2019_Q4.zip")
    df_2020_Q1 = dfs.get("data/Divvy_Trips_2020_Q1.zip")
    
    print("First ZIP DataFrame:")
    
    print("Second ZIP DataFrame:")
    
    Q1. What is the average trip duration per day?
    def avg_trip_duration_per_day(df):
        df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
        result = df.withColumn("date", to_date(col("start_time"))) \
                   .groupBy("date") \
                   .agg(round(avg("Trip_Duration"),1).alias("avg_trip_duration"))
        
        result.write.csv(f"reports/Avg_trip_Duration_PerDay", header=True, mode="overwrite")
        return "Result File for Question Downloaded Successfully!!"
    
    
    # Q2. How many trips were taken each day?
    def total_trips(df):
        df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
        df = df.withColumn("date_only", to_date(col("start_time")))
        result = df.groupBy("date_only").agg(count("trip_id").alias("total_trips"))
    
        result.write.csv(f"reports/TotalTrips_rips_PerDay", header=True, mode="overwrite")
        return "Result File for Question 2 Downloaded Successfully!!"
    
        
    # Q3. What was the most popular starting trip station for each month?
    def Most_Popular_starting_station_each_month(df):
        df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
        df1 = df.withColumn("Month", month(col("start_time")))
        df2 = df1.groupBy("Month", "from_station_name").count()
        df3 = df2.withColumn("Rank", row_number().over(Window.partitionBy("Month").orderBy(col("count").desc())))
        result = df3.filter(col("Rank")==1).select("Month", "from_station_name", "count")
    
        result.write.csv(f"reports/Popular_Station_PerMonth", header=True, mode="overwrite")
        return "Result File for Question 3 Downloaded Successfully!!"
    
    # Q4. What were the top 3 trip stations each day for the last two weeks?
    def top3_trip_stations_each_day_for_last_2_weeks(df):
        df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
        df = df.withColumn("date", to_date("start_time"))
        
        max_date = df.select(F.max('date').alias('max_date')).first()[0]
        two_weeks_before = max_date-timedelta(weeks=2)
    
        filter_data = df.filter((col("date") > two_weeks_before) & (col("date") <= max_date)) \
                    .groupBy("date", "from_station_name") \
                    .count()
    
        window = Window.partitionBy("Date").orderBy(col("count").desc())
    
    
        result = filter_data.withColumn("rank", row_number().over(window)) \
                        .filter(col("rank") <= 3) \
                        .orderBy("date", "rank")
    
        result.write.csv(f"reports/top3_trip_stations_last_2_week", header=True, mode="overwrite")
        return "Result File for Question 4 Downloaded Successfully!!"
    
    
    # Q5. Do Males or Females take longer trips on average?
    
    def trips_avg_of_male_female(df):
        updated_df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
        result = updated_df.groupBy("gender").agg(round(avg("Trip_Duration"),1).alias("Average"))
    
        result.write.csv(f"reports/trips_avg_of_male_female", header=True, mode="overwrite")
        return "Result File for Question 5 Downloaded Successfully!!"
    
    # Q6. What is the top 10 ages of those that take the longest trips, and shortest?
    
    def top_ten_ages_with_shortest_and_longest_trips(df):
        df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
        df = df.withColumn("Year_of_Birth", to_date("birthyear"))
        df = df.withColumn("Age", floor(date_diff(F.current_date(), F.col("Year_of_Birth"))/365.25))
    
        # Store the results in variables without showing them yet
        longest_trips  = df.filter(F.col("Age").isNotNull()).orderBy(col("Trip_Duration").desc()).select("Age", "Trip_Duration").limit(10)
        shortest_trips = df.filter(F.col("Age").isNotNull()).orderBy(col("Trip_Duration").asc()).select("Age", "Trip_Duration").limit(10)
    
        longest_trips.write.csv(f"reports/Top_Ten_Ages/Longest_trip", header=True, mode="overwrite")
    
        shortest_trips.write.csv(f"reports/Top_Ten_Ages/Shortest_trip", header=True, mode="overwrite")
        
        return "Result File for Question 6 Downloaded Successfully!!"

if __name__ == "__main__":
    main()

25/11/06 12:48:37 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Reading file: Divvy_Trips_2019_Q4.csv from data/Divvy_Trips_2019_Q4.zip


25/11/06 12:48:38 WARN TaskSetManager: Stage 54 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe
25/11/06 12:48:38 WARN TaskSetManager: Stage 55 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Reading file: Divvy_Trips_2020_Q1.csv from data/Divvy_Trips_2020_Q1.zip


25/11/06 12:48:41 WARN TaskSetManager: Stage 56 contains a task of very large size (17622 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe
25/11/06 12:48:41 WARN TaskSetManager: Stage 57 contains a task of very large size (17622 KiB). The maximum recommended task size is 1000 KiB.
[Stage 57:>                                                         (0 + 4) / 4]

First ZIP DataFrame:
Second ZIP DataFrame:


Signature: day(col: 'ColumnOrName') -> pyspark.sql.column.Column
Docstring:
Extract the day of the month of a given date/timestamp as integer.

.. versionadded:: 3.5.0

Parameters
----------
col : :class:`~pyspark.sql.Column` or column name
    target date/timestamp column to work on.

Returns
-------
:class:`~pyspark.sql.Column`
    day of the month for given date/timestamp as integer.

See Also
--------
:meth:`pyspark.sql.functions.year`
:meth:`pyspark.sql.functions.quarter`
:meth:`pyspark.sql.functions.month`
:meth:`pyspark.sql.functions.hour`
:meth:`pyspark.sql.functions.minute`
:meth:`pyspark.sql.functions.second`
:meth:`pyspark.sql.functions.dayname`
:meth:`pyspark.sql.functions.dayofyear`
:meth:`pyspark.sql.functions.dayofmonth`
:meth:`pyspark.sql.functions.dayofweek`
:meth:`pyspark.sql.functions.extract`
:meth:`pyspark.sql.functions.datepart`
:meth:`pyspark.sql.functions.date_part`

Examples
--------
Example 1: Extract the day of the month from a string column representing date

In [144]:
from pyspark.sql import SparkSession


def main():
    spark = SparkSession.builder.appName("Exercise6").enableHiveSupport().getOrCreate()
    # your code here


if __name__ == "__main__":
    main()

    avg_trip_duration_per_day(df1)
    total_trips(df1)
    Most_Popular_starting_station_each_month(df1)
    top3_trip_stations_each_day_for_last_2_weeks(df1)
    trips_avg_of_male_female(df1)
    top_ten_ages_with_shortest_and_longest_trips(df1)

25/11/06 12:44:01 WARN TaskSetManager: Stage 19 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
25/11/06 12:44:04 WARN TaskSetManager: Stage 22 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
25/11/06 12:44:06 WARN TaskSetManager: Stage 25 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
25/11/06 12:44:08 WARN TaskSetManager: Stage 31 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
25/11/06 12:44:10 WARN TaskSetManager: Stage 34 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
25/11/06 12:44:12 WARN TaskSetManager: Stage 47 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
25/11/06 12:44:15 WARN TaskSetManager: Stage 50 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.

'Result File for Question 6 Downloaded Successfully!!'

In [4]:
import zipfile
import io
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import timedelta

def main():

    spark = SparkSession.builder.appName("Exercise6").enableHiveSupport().getOrCreate()
    
    zip_files = [
        "data/Divvy_Trips_2019_Q4.zip",
        "data/Divvy_Trips_2020_Q1.zip"
    ]
    
    dfs = {}   
    
    for zip_file in zip_files:
        with zipfile.ZipFile(zip_file, "r") as z:
            for file in z.namelist():
                # Skip macOS or non-CSV files
                if file.startswith("__MACOSX") or not file.endswith(".csv"):
                    continue
    
                print(f"Reading file: {file} from {zip_file}")
    
                # Read file safely (handle encoding issues)
                data = z.read(file).decode("utf-8", errors="ignore")
    
                # Convert the text to Spark RDD
                rows = [row for row in data.split("\n") if row.strip() != ""]
                rdd = spark.sparkContext.parallelize(rows)
    
                # Load as DataFrame
                df = spark.read.csv(rdd, header=True, inferSchema=True)
    
                # Save this dataframe with a name
                dfs[zip_file] = df
    
    # Access DataFrames separately:
    df1 = dfs.get("data/Divvy_Trips_2019_Q4.zip")
    df_2020_Q1 = dfs.get("data/Divvy_Trips_2020_Q1.zip")
    
    print("First ZIP DataFrame:")
    
    print("Second ZIP DataFrame:")
    
    # Q1. What is the average trip duration per day?
    def avg_trip_duration_per_day(df):
        df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
        result = df.withColumn("date", to_date(col("start_time"))) \
                   .groupBy("date") \
                   .agg(round(avg("Trip_Duration"),1).alias("avg_trip_duration"))
        
        result.write.csv(f"reports/Avg_trip_Duration_PerDay", header=True, mode="overwrite")
        return "Result File for Question 1 Downloaded Successfully!!"
    
    
    # Q2. How many trips were taken each day?
    def total_trips(df):
        df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
        df = df.withColumn("date_only", to_date(col("start_time")))
        result = df.groupBy("date_only").agg(count("trip_id").alias("total_trips"))
    
        result.write.csv(f"reports/TotalTrips_rips_PerDay", header=True, mode="overwrite")
        return "Result File for Question 2 Downloaded Successfully!!"
    
        
    # Q3. What was the most popular starting trip station for each month?
    def Most_Popular_starting_station_each_month(df):
        df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
        df1 = df.withColumn("Month", month(col("start_time")))
        df2 = df1.groupBy("Month", "from_station_name").count()
        df3 = df2.withColumn("Rank", row_number().over(Window.partitionBy("Month").orderBy(col("count").desc())))
        result = df3.filter(col("Rank")==1).select("Month", "from_station_name", "count")
    
        result.write.csv(f"reports/Popular_Station_PerMonth", header=True, mode="overwrite")
        return "Result File for Question 3 Downloaded Successfully!!"
    
    # Q4. What were the top 3 trip stations each day for the last two weeks?
    def top3_trip_stations_each_day_for_last_2_weeks(df):
        df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
        df = df.withColumn("date", to_date("start_time"))
        
        max_date = df.select(F.max('date').alias('max_date')).first()[0]
        two_weeks_before = max_date-timedelta(weeks=2)
    
        filter_data = df.filter((col("date") > two_weeks_before) & (col("date") <= max_date)) \
                    .groupBy("date", "from_station_name") \
                    .count()
    
        window = Window.partitionBy("Date").orderBy(col("count").desc())
    
    
        result = filter_data.withColumn("rank", row_number().over(window)) \
                        .filter(col("rank") <= 3) \
                        .orderBy("date", "rank")
    
        result.write.csv(f"reports/top3_trip_stations_last_2_week", header=True, mode="overwrite")
        return "Result File for Question 4 Downloaded Successfully!!"
    
    
    # Q5. Do Males or Females take longer trips on average?
    
    def trips_avg_of_male_female(df):
        updated_df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
        result = updated_df.groupBy("gender").agg(round(avg("Trip_Duration"),1).alias("Average"))
    
        result.write.csv(f"reports/trips_avg_of_male_female", header=True, mode="overwrite")
        return "Result File for Question 5 Downloaded Successfully!!"
    
    # Q6. What is the top 10 ages of those that take the longest trips, and shortest?
    
    def top_ten_ages_with_shortest_and_longest_trips(df):
        df = df.withColumn("Trip_Duration", regexp_replace(col("tripduration"), ",", ""))
        df = df.withColumn("Year_of_Birth", to_date("birthyear"))
        df = df.withColumn("Age", floor(date_diff(F.current_date(), F.col("Year_of_Birth"))/365.25))
    
        # Store the results in variables without showing them yet
        longest_trips  = df.filter(F.col("Age").isNotNull()).orderBy(col("Trip_Duration").desc()).select("Age", "Trip_Duration").limit(10)
        shortest_trips = df.filter(F.col("Age").isNotNull()).orderBy(col("Trip_Duration").asc()).select("Age", "Trip_Duration").limit(10)
    
        longest_trips.write.csv(f"reports/Top_Ten_Ages/Longest_trip", header=True, mode="overwrite")
    
        shortest_trips.write.csv(f"reports/Top_Ten_Ages/Shortest_trip", header=True, mode="overwrite")
        
        return "Result File for Question 6 Downloaded Successfully!!"

    a = avg_trip_duration_per_day(df1)
    b = total_trips(df1)
    c = Most_Popular_starting_station_each_month(df1)
    d = top3_trip_stations_each_day_for_last_2_weeks(df1)
    e = trips_avg_of_male_female(df1)
    f = top_ten_ages_with_shortest_and_longest_trips(df1)
    
    print(a)
    print(b)
    print(c)
    print(d)
    print(e)
    print(f)

if __name__ == "__main__":
    main()

Reading file: Divvy_Trips_2019_Q4.csv from data/Divvy_Trips_2019_Q4.zip


25/11/06 15:03:25 WARN TaskSetManager: Stage 0 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>                   (0 + 1) / 1]
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe
25/11/06 15:03:26 WARN TaskSetManager: Stage 1 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Reading file: Divvy_Trips_2020_Q1.csv from data/Divvy_Trips_2020_Q1.zip


25/11/06 15:03:30 WARN TaskSetManager: Stage 2 contains a task of very large size (17622 KiB). The maximum recommended task size is 1000 KiB.
Exception ignored in: <_io.BufferedWriter name=5>
Traceback (most recent call last):
  File "/home/developer/anaconda3/lib/python3.13/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe
25/11/06 15:03:30 WARN TaskSetManager: Stage 3 contains a task of very large size (17622 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

First ZIP DataFrame:
Second ZIP DataFrame:


25/11/06 15:03:32 WARN TaskSetManager: Stage 4 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
25/11/06 15:03:36 WARN TaskSetManager: Stage 7 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
25/11/06 15:03:38 WARN TaskSetManager: Stage 10 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
25/11/06 15:03:41 WARN TaskSetManager: Stage 16 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
25/11/06 15:03:44 WARN TaskSetManager: Stage 19 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
25/11/06 15:03:46 WARN TaskSetManager: Stage 32 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
25/11/06 15:03:49 WARN TaskSetManager: Stage 35 contains a task of very large size (24036 KiB). The maximum recommended task size is 1000 KiB.
2

Result File for Question 1 Downloaded Successfully!!
Result File for Question 2 Downloaded Successfully!!
Result File for Question 3 Downloaded Successfully!!
Result File for Question 4 Downloaded Successfully!!
Result File for Question 5 Downloaded Successfully!!
Result File for Question 6 Downloaded Successfully!!


In [5]:
spark.stop()